# Combining the flatfox and ImmoScout24 rental apartment data (buildings built before 1990)

**This Jupyter notebook does the following:**

- Imports the two prepared data sets `Data/flatfox_pre1991_prepared.csv` and `Data/immoscout24_pre1990_prepared.csv`
- Selects the same columns in both data sets and combines them into one data frame
- Finds apartments that are advertised on **both** platforms and keeps them only once
- Counts the apartments per renovation group (energy renovation / general renovation / not renovated)
- Saves the combined data to `Data/apartments_pre1990_combined.csv`

Run `02_flatfox_data_preparation.ipynb` and `03_immoscout24_data_preparation.ipynb` first.

## Libraries and settings

In [1]:
# Libraries
import os
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

c:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Importing data

In [2]:
# Read the prepared data sets
df_ff = pd.read_csv('./Data/flatfox_pre1991_prepared.csv', sep=',', encoding='utf-8')
df_is = pd.read_csv('./Data/immoscout24_pre1990_prepared.csv', sep=',', encoding='utf-8')

print('flatfox:', df_ff.shape)
print('ImmoScout24:', df_is.shape)

flatfox: (391, 34)
ImmoScout24: (387, 32)


## Select and rename the same columns in both data sets

The two platforms use different column names. Both data frames get the same columns, so that they can be stacked with `pd.concat()`.

- flatfox has no street address, no floor and no shared rooms; these columns stay empty or `False`.
- ImmoScout24 has no weekly rents; `rent_per_week` is `False`.
- The flatfox net rent (`rent_net`) comes from the flatfox Web API.

In [3]:
cols = ['source', 'listing_url', 'address', 'zip', 'locality', 'rooms', 'area_clean',
        'price', 'price_net', 'rent_per_m2', 'rent_net_per_m2',
        'year_built_clean', 'year_renovated_clean',
        'furnished', 'temporary', 'shared', 'rent_per_week',
        'keyword_energy', 'keyword_general', 'renovation_group', 'minergie', 'description']

# flatfox
ff = df_ff.rename(columns={'detail_url-href': 'listing_url', 'rent_net': 'price_net'})
ff['source'] = 'flatfox'
ff['shared'] = False
ff['rent_net_per_m2'] = (ff['price_net'] / ff['area_clean']).where(ff['rent_period'].eq('month')).round(2)
ff = ff[cols]

# ImmoScout24
im = df_is.rename(columns={'listing-href': 'listing_url'})
im['source'] = 'immoscout24'
im['rent_per_week'] = False
im = im[cols]

print(ff.shape, im.shape)
ff.head()

(391, 22) (387, 22)


,source,listing_url,address,zip,locality,rooms,area_clean,price,price_net,rent_per_m2,...,year_renovated_clean,furnished,temporary,shared,rent_per_week,keyword_energy,keyword_general,renovation_group,minergie,description
0,flatfox,https://flatfox.ch/en/flat/6614-brissago/1848132/,6614 Brissago,6614,Brissago,3.0,NaN,1200.0,1200.0,NaN,...,2011.0,False,False,False,False,False,False,general renovation,False,"Brissago: Appartamento 3 locali\n\nA Brissago,..."
1,flatfox,https://flatfox.ch/en/flat/neufeldstrasse-7-94...,9450 Altstätten,9450,Altstätten,4.0,81.0,1460.0,1230.0,18.02,...,NaN,False,False,False,False,False,False,not renovated,False,Erster Nettomietzins geschenkt! Herzlich Willk...
2,flatfox,https://flatfox.ch/en/flat/rue-de-la-chapelle-...,2610 Saint-Imier,2610,Saint-Imier,3.0,60.0,1050.0,900.0,17.50,...,2025.0,False,False,False,False,False,True,general renovation,False,Charmant appartement de 3 pièces au cœur de Sa...
3,flatfox,https://flatfox.ch/en/flat/ringgasse-27-9450-a...,9450 Altstätten,9450,Altstätten,3.5,78.0,1395.0,1265.0,17.88,...,2009.0,False,False,False,False,False,False,general renovation,False,MIETZINSFREIE ZEIT GENIESSEN - TOLLE 3.5 ZIMME...
4,flatfox,https://flatfox.ch/en/flat/ch-de-la-redoute-36...,1260 Nyon,1260,Nyon,3.0,80.0,2100.0,1900.0,26.25,...,2010.0,False,False,False,False,False,False,general renovation,False,A louer sans caution - Joli logement bien situ...


## Find apartments that are advertised on both platforms

Landlords often advertise the same apartment on flatfox **and** ImmoScout24. The listing URLs are different, so another key is needed. The flatfox data contains only the postcode (no street), so the street cannot be used.

An apartment is treated as the **same** apartment if all five values are identical:

| Variable | Why |
|---|---|
| `zip` | same place |
| `rooms` | same size |
| `area_clean` | same living area in m² |
| `price` | same gross rent in CHF |
| `year_built_clean` | same building |

Using five variables together makes it very unlikely that two *different* apartments are matched by chance. Listings with a missing value in one of these variables are not matched (they are kept).

The matching is **one-to-one**: if a building has two identical flats on one platform and one on the other, only one pair is matched. This is done with a counter (`cumcount()`) inside each key.

In [4]:
key = ['zip', 'rooms', 'area_clean', 'price', 'year_built_clean']

# Counter within each key (0, 1, 2, ...), so that the matching is one-to-one
ff['n_in_key'] = ff.groupby(key, dropna=False).cumcount()
im['n_in_key'] = im.groupby(key, dropna=False).cumcount()

# Pairs: same key and same counter (rows with a missing key value are excluded)
pairs = (ff.dropna(subset=key).reset_index()
           .merge(im.dropna(subset=key).reset_index(), on=key + ['n_in_key'], suffixes=('_ff', '_is')))

print('Apartments advertised on both platforms:', len(pairs), '\n')

# Show the pairs
pairs[key + ['address_is', 'renovation_group_ff', 'renovation_group_is']]

Apartments advertised on both platforms: 14 



,zip,rooms,area_clean,price,year_built_clean,address_is,renovation_group_ff,renovation_group_is
0,8750,2.5,62.0,1250.0,1861.0,"Kirchstr. / Postgasse 23 16/po, 8750 Glarus",not renovated,not renovated
1,6600,4.0,96.0,1560.0,1973.0,"Via Franzoni 31, 6600 Locarno",not renovated,not renovated
2,1680,4.5,99.0,2120.0,1986.0,"Pré de la Grange 4, 1680 Romont FR",not renovated,not renovated
3,3550,3.0,62.0,1216.0,1964.0,"Napfstr. 2, 3550 Langnau im Emmental",not renovated,not renovated
4,1950,4.5,92.0,1600.0,1979.0,"Avenue Maurice-Troillet 79, 1950 Sion",not renovated,not renovated
5,6600,2.5,116.0,1560.0,1899.0,"Via Ramogna 8, 6600 Locarno",general renovation,general renovation
6,6500,3.5,65.0,1250.0,1985.0,"Via Franco Zorzi 25a, 6500 Bellinzona",energy renovation,energy renovation
7,3930,4.5,100.0,1900.0,1965.0,"Kleegärtenstrasse 17, 3930 Visp",general renovation,general renovation
8,2013,3.5,82.0,2100.0,1988.0,"Ch. des Sources 16, 2013 Colombier NE",energy renovation,energy renovation
9,8750,2.5,71.0,1335.0,1861.0,"Kirchstr. / Postgasse 23 16/po, 8750 Glarus",not renovated,not renovated


### Check: do both platforms give the same renovation group?

If the groups differ, the combined row gets the group with **more** information: a keyword or a renovation year found on one platform also counts for the combined row.

In [5]:
same_group = pairs['renovation_group_ff'] == pairs['renovation_group_is']
print('Same renovation group:', same_group.sum(), 'of', len(pairs), '\n')

# Pairs with different groups (if any)
pairs.loc[~same_group, key + ['renovation_group_ff', 'renovation_group_is']]

Same renovation group: 14 of 14 



,zip,rooms,area_clean,price,year_built_clean,renovation_group_ff,renovation_group_is


### Check with the street address

The flatfox column `address` has only the postcode, but the listing URL contains the street, e.g. `.../flat/napfstr-2-3550-langnau-im-emmental/86343716/`. The street is taken from the URL and compared with the ImmoScout24 street (lower case, without accents, spaces and punctuation, "strasse" = "str").

- Pairs with the same address **and** the same five key values are the same apartment.
- Pairs with the same address but different rooms, area or rent are **different** flats in the same building. They must not be removed, so the address alone is not a good key.

In [6]:
import re
import unicodedata

def norm_street(s):
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKD', s.lower())
    s = ''.join(c for c in s if not unicodedata.combining(c)).replace('ß', 'ss')
    s = re.sub(r'[^a-z0-9]', '', s)      # remove spaces, hyphens, dots, ...
    return s.replace('strasse', 'str')

# flatfox: street = URL slug before the postcode
slug = ff['listing_url'].str.extract(r'/flat/([^/]+)/\d+/?$', expand=False)
ff['street'] = [norm_street(re.sub(r'-?%s.*$' % z, '', s)) if isinstance(s, str) else ''
                for s, z in zip(slug, ff['zip'].astype(str))]

# ImmoScout24: street = part of the address before ', 1234'
im['street'] = im['address'].str.extract(r'^(.*),\s*\d{4}', expand=False).map(norm_street)

print('flatfox with street:', (ff['street'] != '').sum(), 'of', len(ff))
print('ImmoScout24 with street:', (im['street'] != '').sum(), 'of', len(im), '\n')

# Pairs with the same postcode and street
same_address = (ff[ff['street'] != ''].reset_index()
                .merge(im[im['street'] != ''].reset_index(), on=['zip', 'street'], suffixes=('_ff', '_is')))
same_address['same_key'] = np.logical_and.reduce(
    [same_address[c + '_ff'] == same_address[c + '_is'] for c in ['rooms', 'area_clean', 'price', 'year_built_clean']])

print('Buildings with the same address on both platforms:', same_address[['zip', 'street']].drop_duplicates().shape[0])
print('Pairs with the same address:', len(same_address))
print('Matched apartments (key) that also have the same address:',
      pairs['index_ff'].isin(same_address['index_ff']).sum(), 'of', len(pairs), '\n')

# Same address, but a different flat (not removed)
same_address.loc[~same_address['same_key'],
                 ['address_is', 'rooms_ff', 'rooms_is', 'area_clean_ff', 'area_clean_is', 'price_ff', 'price_is']]

flatfox with street: 331 of 391
ImmoScout24 with street: 376 of 387 

Buildings with the same address on both platforms: 16
Pairs with the same address: 22
Matched apartments (key) that also have the same address: 14 of 14 



,address_is,rooms_ff,rooms_is,area_clean_ff,area_clean_is,price_ff,price_is
0,"Kirchstr. / Postgasse 23 16/po, 8750 Glarus",2.5,2.5,62.0,71.0,1250.0,1335.0
4,"Remisbergstr. 47, 8280 Kreuzlingen",4.5,4.5,96.0,83.0,1940.0,1775.0
5,"Via Carlo Pasta 3, 6850 Mendrisio",4.5,3.5,120.0,100.0,1630.0,1500.0
9,"Via Franco Zorzi 25a, 6500 Bellinzona",3.5,3.5,72.0,65.0,1330.0,1250.0
15,"Kirchstr. / Postgasse 23 16/po, 8750 Glarus",2.5,2.5,71.0,62.0,1335.0,1250.0
16,"Ch. des Sources 16a, 2013 Colombier NE",3.5,3.5,68.0,81.0,2130.0,2100.0
19,"Ebenaustr. 15, 6048 Horw",3.0,4.0,70.0,85.0,1607.0,2145.0


## Combine the two data sets and remove the duplicated apartments

- For an apartment on both platforms, the **ImmoScout24 row is kept**, because it also has the street address and the floor. Its `source` becomes `'both'`.
- The information of the flatfox row is added: keywords and `minergie` are combined with "or", and missing values (e.g. net rent, renovation year) are filled from flatfox.
- The flatfox row is then removed.

In [7]:
im_c = im.copy()
idx_is = pairs['index_is'].values
idx_ff = pairs['index_ff'].values

# Mark the apartments that are on both platforms
im_c.loc[idx_is, 'source'] = 'both'

# Combine keywords and Minergie with 'or'
for c in ['keyword_energy', 'keyword_general']:
    im_c.loc[idx_is, c] = im_c.loc[idx_is, c].values | ff.loc[idx_ff, c].values
im_c['minergie'] = im_c['minergie'].astype('boolean')
im_c.loc[idx_is, 'minergie'] = (im_c.loc[idx_is, 'minergie'].fillna(False).values
                                | ff.loc[idx_ff, 'minergie'].astype('boolean').fillna(False).values)

# Fill missing values from the flatfox row
for c in ['price_net', 'rent_net_per_m2', 'year_renovated_clean']:
    missing = im_c.loc[idx_is, c].isna().values
    im_c.loc[idx_is[missing], c] = ff.loc[idx_ff[missing], c].values

# Stack both data sets without the matched flatfox rows
df = pd.concat([im_c, ff.drop(index=idx_ff)], ignore_index=True).drop(columns=['n_in_key', 'street'])
df['minergie'] = df['minergie'].astype('boolean')

print('flatfox + ImmoScout24:', len(ff) + len(im))
print('Removed (on both platforms):', len(idx_ff))
print('Combined data:', df.shape, '\n')
print(df['source'].value_counts())

flatfox + ImmoScout24: 778
Removed (on both platforms): 14
Combined data: (764, 22) 

source
flatfox        377
immoscout24    373
both            14
Name: count, dtype: int64


### Check: construction year

All apartments with a known construction year were built **before 1990** (apartments built in 1990 were removed in both preparation notebooks). For some flatfox listings the construction year is unknown (missing or an invalid value like `0`); they are kept, because the scrape filter only shows buildings up to 1990.

In [8]:
print('Built before 1990:', df['year_built_clean'].le(1989).sum())
print('Built in 1990 or later:', df['year_built_clean'].ge(1990).sum())
print('Construction year unknown:', df['year_built_clean'].isna().sum(), '\n')

print(df['year_built_clean'].describe().round(0))

Built before 1990: 749
Built in 1990 or later: 0
Construction year unknown: 15 

count     749.0
mean     1946.0
std        60.0
min      1478.0
25%      1932.0
50%      1965.0
75%      1977.0
max      1989.0
Name: year_built_clean, dtype: float64


## Renovation groups in the combined data

The group is assigned again with the same rule as in the two preparation notebooks, so that the combined keywords and renovation years are used:

1. **energy renovation**: energy keyword in the description
2. **general renovation**: general keyword **or** a valid renovation year
3. **not renovated**: none of the above

In [9]:
df['renovation_group'] = np.select(
    [df['keyword_energy'].astype(bool),
     df['keyword_general'].astype(bool) | df['year_renovated_clean'].notna()],
    ['energy renovation', 'general renovation'],
    default='not renovated')

# Number and share of apartments per renovation group
groups = df['renovation_group'].value_counts().to_frame('count')
groups['percent'] = (groups['count'] / groups['count'].sum() * 100).round(1)
groups.loc['total'] = [groups['count'].sum(), 100.0]
groups

,count,percent
renovation_group,,
general renovation,509.0,66.6
not renovated,208.0,27.2
energy renovation,47.0,6.2
total,764.0,100.0


In [10]:
# Renovation groups by source
pd.crosstab(df['renovation_group'], df['source'], margins=True)

source,both,flatfox,immoscout24,All
renovation_group,,,,
energy renovation,2,16,29,47
general renovation,6,234,269,509
not renovated,6,127,75,208
All,14,377,373,764


In [11]:
# Gross rent per m² by renovation group (first descriptive comparison, no controls)
df.groupby('renovation_group')['rent_per_m2'].agg(['count', 'mean', 'median']).round(2)

,count,mean,median
renovation_group,,,
energy renovation,46,24.47,23.44
general renovation,479,24.69,22.24
not renovated,194,24.03,21.13


In [12]:
# Minergie by renovation group
pd.crosstab(df['renovation_group'], df['minergie'].astype('str'), margins=True)

minergie,False,True,All
renovation_group,,,
energy renovation,40,7,47
general renovation,505,4,509
not renovated,208,0,208
All,753,11,764


### Save combined data to file (before the final cleaning)

In [13]:
df.to_csv('./Data/apartments_pre1990_combined.csv',
          sep=",",
          encoding='utf-8',
          index=False)

print('Saved:', df.shape[0], 'apartments')

Saved: 764 apartments


## Final cleaning: duplicates, missing rents and outliers

The following steps create the data set for the analysis (`Data/apartments_pre1990_clean.csv`). The number of rows is printed after each step. The thresholds follow the approach of `Week_10/exploratory_data_analysis_apartment_data.ipynb` (own thresholds, with a justification).

### Invalid net rents

A net rent of 0 CHF is not possible (error in the flatfox Web API data). It is set to missing; the row is kept because the gross rent is known.

In [14]:
print('Net rent = 0:', df['price_net'].eq(0).sum())

df.loc[df['price_net'].eq(0), ['price_net', 'rent_net_per_m2']] = np.nan

df_clean = df.copy()
print('Number of rows:', df_clean.shape[0])

Net rent = 0: 1
Number of rows: 764


### Duplicated values

- Same listing URL: the same listing twice.
- Same postcode, gross rent and description: the same advertisement posted twice under a different URL.
- Same postcode, rooms, area, rent and construction year on the **same** platform: shown for checking only. These are different flats in the same building (different floor or house number), so they are **kept**.

In [15]:
print('Number of rows before removing duplicates:', df_clean.shape[0])

dup_url = df_clean.duplicated(subset=['listing_url'])
dup_text = df_clean['description'].notna() & df_clean.duplicated(subset=['zip', 'price', 'description'])
print('Duplicated listing URLs:', dup_url.sum())
print('Duplicated advertisements (postcode, rent, description):', dup_text.sum())

df_clean = df_clean.loc[~(dup_url | dup_text)].reset_index(drop=True)
print('Number of rows after removing duplicates:', df_clean.shape[0], '\n')

# Check only: identical flats in the same building (kept)
key = ['zip', 'rooms', 'area_clean', 'price', 'year_built_clean']
same_flat = df_clean[key].notna().all(axis=1) & df_clean.duplicated(subset=key, keep=False)
df_clean.loc[same_flat, ['source', 'address'] + key].sort_values(key)

Number of rows before removing duplicates: 764
Duplicated listing URLs: 0
Duplicated advertisements (postcode, rent, description): 0
Number of rows after removing duplicates: 764 



,source,address,zip,rooms,area_clean,price,year_built_clean
40,immoscout24,"Ch. de Monséjour 19, 1700 Fribourg",1700,3.5,72.0,1705.0,1961.0
347,immoscout24,"Ch. de Monséjour 19, 1700 Fribourg",1700,3.5,72.0,1705.0,1961.0
43,both,"Ch. des Sources 16, 2013 Colombier NE",2013,3.5,82.0,2100.0,1988.0
55,immoscout24,"Ch. des Sources 16, 2013 Colombier NE",2013,3.5,82.0,2100.0,1988.0
73,immoscout24,"Heidenstr. 18, 4142 Münchenstein",4142,2.5,87.0,1800.0,1989.0
119,immoscout24,"Heidenstr. 20, 4142 Münchenstein",4142,2.5,87.0,1800.0,1989.0


### Missing rents

Rows without a gross rent ("price on request") cannot be used to analyse rents and are removed. The one listing with a **weekly** rent is also removed, because its rent cannot be compared with monthly rents.

In [16]:
print('Number of rows before removing missing rents:', df_clean.shape[0])
print('Missing gross rent:', df_clean['price'].isna().sum())
print('Weekly rent:', df_clean['rent_per_week'].sum())

df_clean = df_clean.loc[df_clean['price'].notna() & ~df_clean['rent_per_week']].reset_index(drop=True)
print('Number of rows after removing missing rents:', df_clean.shape[0])

Number of rows before removing missing rents: 764
Missing gross rent: 16
Weekly rent: 1
Number of rows after removing missing rents: 747


### Outliers

First, the distribution of rent, area and rent per m² is shown with quantiles. Then the following thresholds are used:

| Condition | Justification |
|---|---|
| `shared` is `True` | a room in a shared flat (WG) is not an apartment |
| `area_clean` < 20 m² | single rooms or wrong areas, not comparable with apartments |
| `rent_per_m2` < 10 CHF | far below the Swiss rent level, very likely an error |
| `rent_per_m2` > 60 CHF | holiday, serviced or luxury flats with services included, not a normal rental market |

Rows with a missing area are **kept** (they cannot be checked; they can still be used for analyses without the area). Note that unusual apartments (e.g. very large or expensive ones) are not automatically errors, so the thresholds are kept wide. Expensive cities like Zürich and Geneva often have rents of 35–50 CHF/m², which is why a statistical rule like 1.5 × IQR (upper bound ≈ 39 CHF/m²) would remove too many normal city apartments.

In [17]:
df_clean[['price', 'area_clean', 'rent_per_m2']].quantile(q=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).round(2)

,price,area_clean,rent_per_m2
0.01,784.6,22.0,12.77
0.05,1050.0,38.0,14.99
0.25,1497.5,65.0,18.94
0.50,1800.0,81.0,22.19
0.75,2230.0,100.0,27.00
0.95,3570.0,145.0,42.87
0.99,5060.2,186.4,60.14


In [18]:
print('Number of rows before removing outliers:', df_clean.shape[0])

outlier = (df_clean['shared']
           | df_clean['area_clean'].lt(20)
           | df_clean['rent_per_m2'].lt(10)
           | df_clean['rent_per_m2'].gt(60))

print('Shared rooms:', df_clean['shared'].sum())
print('Area < 20 m²:', df_clean['area_clean'].lt(20).sum())
print('Rent per m² < 10 CHF:', df_clean['rent_per_m2'].lt(10).sum())
print('Rent per m² > 60 CHF:', df_clean['rent_per_m2'].gt(60).sum(), '\n')

# Show the removed outliers
display(df_clean.loc[outlier, ['source', 'address', 'rooms', 'area_clean', 'price', 'rent_per_m2']])

df_clean = df_clean.loc[~outlier].reset_index(drop=True)
print('Number of rows after removing outliers:', df_clean.shape[0])

Number of rows before removing outliers: 747
Shared rooms: 2
Area < 20 m²: 4
Rent per m² < 10 CHF: 1
Rent per m² > 60 CHF: 8 



,source,address,rooms,area_clean,price,rent_per_m2
70,immoscout24,"Rheingasse 25, 4058 Basel",3.5,61.0,3800.0,62.30
118,immoscout24,"Rheingasse 25, 4058 Basel",2.5,37.0,2500.0,67.57
127,immoscout24,"Rue Liotard 77, 1203 Genève",3.0,45.0,2950.0,65.56
178,immoscout24,"Gundeldingerstrasse, 4053 Basel",1.0,20.0,775.0,NaN
207,immoscout24,"Bernstrasse 26, 3360 Herzogenbuchsee",1.0,12.0,655.0,54.58
245,immoscout24,"Oberdorf 37, 3207 Wileroltigen",1.0,25.0,700.0,NaN
280,immoscout24,7031 Laax GR,1.5,30.0,2100.0,70.00
287,immoscout24,"Eisenbahnstrasse 9, 4900 Langenthal",1.0,164.0,1600.0,9.76
360,immoscout24,"Erlenweg 4, 8702 Zollikon",2.5,65.0,3950.0,60.77
515,flatfox,3963 Crans-Montana,1.0,24.0,1900.0,79.17


Number of rows after removing outliers: 733


### Renovation groups after the final cleaning

In [19]:
groups = df_clean['renovation_group'].value_counts().to_frame('count')
groups['percent'] = (groups['count'] / groups['count'].sum() * 100).round(1)
groups.loc['total'] = [groups['count'].sum(), 100.0]
groups['count'] = groups['count'].astype(int)
groups

,count,percent
renovation_group,,
general renovation,486,66.3
not renovated,201,27.4
energy renovation,46,6.3
total,733,100.0


In [20]:
# Gross rent per m² by renovation group (first descriptive comparison, no controls)
df_clean.groupby('renovation_group')['rent_per_m2'].agg(['count', 'mean', 'median']).round(2)

,count,mean,median
renovation_group,,,
energy renovation,46,24.47,23.44
general renovation,469,23.97,22.17
not renovated,192,23.78,21.04


### Save cleaned data to file

In [21]:
df_clean.to_csv('./Data/apartments_pre1990_clean.csv',
                sep=",",
                encoding='utf-8',
                index=False)

print('Saved:', df_clean.shape[0], 'apartments')

Saved: 733 apartments


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [22]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:45:01
Python Version: 3.11.8
-----------------------------------
